# Task 5 — So sánh 2–3 SKU Groups (Fast / Medium / Slow)

**Mục tiêu:** Chứng minh framework mở rộng được xuống product-group level.
- **3 yêu cầu so sánh:** (1) A2C trên 3 groups, (2) DQN trên 3 groups, (3) A2C vs DQN trong từng group — **chỉ so sánh nội bộ 3 groups**, không so với baseline 220.
- **Dữ liệu:** Checkpoint + logs đã train xong tại `Feedback 7-9/task12-9/task 5/outputA2C_*_73` và `outputDQN_*_73` (600 episodes × 900 timesteps, 14 actions, không mock).
- **Nguồn phân nhóm:** `prepare_grouped_task5.ipynb` chia 220 SKU theo MeanDemand → Fast 73 (top), Medium 73 (mid), Slow 74 (bottom, CV cao).


## 0. Setup — paths & imports (đường dẫn tuyệt đối)

In [1]:
import os, json, glob, pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE = r"C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5"
GROUPS = ["Fast", "Medium", "Slow"]
A2C_DIRS = {g: os.path.join(BASE, f"outputA2C_{g}_73") for g in GROUPS}
DQN_DIRS = {g: os.path.join(BASE, f"outputDQN_{g}_73") for g in GROUPS}
DATA_GROUPED = os.path.join(BASE, "data_grouped")

print("BASE:", BASE)
for g in GROUPS:
    print(g, "A2C:", A2C_DIRS[g], "exists:", os.path.isdir(A2C_DIRS[g]))
    print(g, "DQN:", DQN_DIRS[g], "exists:", os.path.isdir(DQN_DIRS[g]))
for g in ["group_fast","group_medium","group_slow"]:
    p = os.path.join(DATA_GROUPED, g)
    print(g, os.listdir(p) if os.path.isdir(p) else "MISSING")


BASE: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5
Fast A2C: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputA2C_Fast_73 exists: True
Fast DQN: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputDQN_Fast_73 exists: True
Medium A2C: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputA2C_Medium_73 exists: True
Medium DQN: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputDQN_Medium_73 exists: True
Slow A2C: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputA2C_Slow_73 exists: True
Slow DQN: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\outputDQN_Slow_73 exists: True
group_fast ['capacity.csv', 'capacity.tfrecords', 'stock.csv', 'stock.tfrecords', 'test.csv', 'test.tfrecords', 'train.csv', 'train.tfrecords']
group_medium ['capacity.csv', 'capacity.tfrecords', 'stock.csv', 'stock

## 1. Load training summaries (600 episodes) — không mock, đọc file thật

- A2C Medium/Slow: `training_summary_*.json` (đã có)
- A2C Fast: không có summary gộp → reconstruct từ `training_log_*_episode_*.json` (639 files, sẽ dedup theo episode → 460 unique, missing 461-600)
- DQN 3 groups: `training_summary_*.json` (reward/stockout/waste/overstock/quantile)


In [2]:
def load_a2c_summary_from_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["reward"] = df["rewards_mean"]
    df["stockout"] = df["stockouts_mean"]
    df["waste"] = df["waste_mean"]
    df["group"] = None
    df["algo"] = "A2C_mod"
    return df

def reconstruct_a2c_fast_from_logs(log_dir):
    files = sorted(glob.glob(os.path.join(log_dir, "training_log_*_episode_*.json")))
    # dedup theo episode: lay file timestamp lon nhat
    ep_to_file = {}
    for fp in files:
        base = os.path.basename(fp)
        ep = int(base.split("episode_")[1].split(".")[0])
        if ep not in ep_to_file or fp > ep_to_file[ep]:
            ep_to_file[ep] = fp
    unique_eps = sorted(ep_to_file.keys())
    missing = [e for e in range(1, 601) if e not in ep_to_file]
    best_ts = max(set(os.path.basename(f).split("_")[2]+"_"+os.path.basename(f).split("_")[3] for f in files)) if files else "N/A"
    print(f"A2C Fast reconstruct: total_files={len(files)} unique_eps={len(unique_eps)} range={unique_eps[0] if unique_eps else 'NA'}-{unique_eps[-1] if unique_eps else 'NA'} missing={len(missing)} (e.g. {missing[:10]}) best_ts={best_ts}")
    if missing:
        print(f"  CANH BAO: Fast thieu {len(missing)} episodes cuoi (461-600) — se danh gia tren {len(unique_eps)} episodes co san")
    best_files = [ep_to_file[ep] for ep in unique_eps]
    rows=[]
    for fp in best_files:
        ep = int(os.path.basename(fp).split("episode_")[1].split(".")[0])
        with open(fp,'r',encoding='utf-8') as f:
            j=json.load(f)
        rewards = np.array(j.get("rewards", []), dtype=float)
        waste = np.array(j.get("waste", []), dtype=float)
        stockouts = np.array(j.get("stockouts", []), dtype=float)
        delta = np.array(j.get("delta", []), dtype=float)
        critic = np.array(j.get("critic_loss", []), dtype=float)
        actor = np.array(j.get("actor_loss", []), dtype=float)
        rows.append({
            "episode": ep,
            "steps": len(rewards),
            "rewards_mean": float(np.mean(rewards)) if len(rewards) else np.nan,
            "rewards_std": float(np.std(rewards)) if len(rewards) else np.nan,
            "stockouts_mean": float(np.mean(stockouts)) if len(stockouts) else np.nan,
            "waste_mean": float(np.mean(waste)) if len(waste) else np.nan,
            "delta_mean": float(np.mean(delta)) if len(delta) else np.nan,
            "critic_loss_mean": float(np.mean(critic)) if len(critic) else np.nan,
            "actor_loss_mean": float(np.mean(actor)) if len(actor) else np.nan,
        })
    df = pd.DataFrame(rows).sort_values("episode").reset_index(drop=True)
    df["reward"] = df["rewards_mean"]
    df["stockout"] = df["stockouts_mean"]
    df["waste"] = df["waste_mean"]
    df["algo"] = "A2C_mod"
    return df, best_ts

def load_dqn_summary(path):
    with open(path,'r',encoding='utf-8') as f:
        data=json.load(f)
    df=pd.DataFrame(data)
    df["algo"]="DQN"
    return df

# --- load A2C Medium/Slow ---
a2c_medium_path = glob.glob(os.path.join(A2C_DIRS["Medium"], "logs", "training_summary*.json"))[0]
a2c_slow_path = glob.glob(os.path.join(A2C_DIRS["Slow"], "logs", "training_summary*.json"))[0]
df_a2c_medium = load_a2c_summary_from_file(a2c_medium_path)
df_a2c_slow = load_a2c_summary_from_file(a2c_slow_path)
df_a2c_fast, fast_ts = reconstruct_a2c_fast_from_logs(os.path.join(A2C_DIRS["Fast"], "logs"))
print("A2C Fast reconstructed episodes:", len(df_a2c_fast), "cols:", df_a2c_fast.columns.tolist()[:8])
print("A2C Medium episodes:", len(df_a2c_medium))
print("A2C Slow episodes:", len(df_a2c_slow))

# --- load DQN ---
dqn_fast_path = glob.glob(os.path.join(DQN_DIRS["Fast"], "logs", "training_summary*.json"))[0]
dqn_med_path = glob.glob(os.path.join(DQN_DIRS["Medium"], "logs", "training_summary*.json"))[0]
dqn_slow_path = glob.glob(os.path.join(DQN_DIRS["Slow"], "logs", "training_summary*.json"))[0]
df_dqn_fast = load_dqn_summary(dqn_fast_path)
df_dqn_med = load_dqn_summary(dqn_med_path)
df_dqn_slow = load_dqn_summary(dqn_slow_path)
print("DQN Fast/med/slow episodes:", len(df_dqn_fast), len(df_dqn_med), len(df_dqn_slow))

# gan group label
for df,g in [(df_a2c_fast,"Fast"),(df_a2c_medium,"Medium"),(df_a2c_slow,"Slow")]:
    df["group"]=g
for df,g in [(df_dqn_fast,"Fast"),(df_dqn_med,"Medium"),(df_dqn_slow,"Slow")]:
    df["group"]=g

df_a2c = pd.concat([df_a2c_fast, df_a2c_medium, df_a2c_slow], ignore_index=True)
df_dqn = pd.concat([df_dqn_fast, df_dqn_med, df_dqn_slow], ignore_index=True)
print("A2C combined:", df_a2c.shape, "DQN combined:", df_dqn.shape)
df_a2c.head(2)


A2C Fast reconstruct: total_files=639 unique_eps=460 range=1-460 missing=140 (e.g. [461, 462, 463, 464, 465, 466, 467, 468, 469, 470]) best_ts=20260917_171517
  CANH BAO: Fast thieu 140 episodes cuoi (461-600) — se danh gia tren 460 episodes co san
A2C Fast reconstructed episodes: 460 cols: ['episode', 'steps', 'rewards_mean', 'rewards_std', 'stockouts_mean', 'waste_mean', 'delta_mean', 'critic_loss_mean']
A2C Medium episodes: 600
A2C Slow episodes: 600
DQN Fast/med/slow episodes: 600 600 600
A2C combined: (1660, 15) DQN combined: (1800, 12)


,episode,steps,rewards_mean,rewards_std,stockouts_mean,waste_mean,delta_mean,critic_loss_mean,actor_loss_mean,reward,stockout,waste,algo,group,entropy_adjusted_mean
0,1,33,0.173417,0.223488,0.155342,0.010270,0.092982,0.059803,1.019842e-06,0.173417,0.155342,0.010270,A2C_mod,Fast,NaN
1,2,33,0.176921,0.221781,0.147001,0.010564,0.096478,0.058665,9.142246e-07,0.176921,0.147001,0.010564,A2C_mod,Fast,NaN


## 2. Kiểm tra checkpoint & data_grouped (minh chứng train thật)

In [3]:
import pathlib
for g, d in A2C_DIRS.items():
    ckpt = os.path.join(d, "checkpoints", "checkpoint")
    if os.path.exists(ckpt):
        print(f"A2C {g}: {open(ckpt,encoding='utf-8').readline().strip()}")
        print("  ckpt files:", len(glob.glob(os.path.join(d,"checkpoints","ckpt-*.index"))))
for g, d in DQN_DIRS.items():
    ckpt = os.path.join(d, "checkpoints", "checkpoint")
    if os.path.exists(ckpt):
        print(f"DQN {g}: {open(ckpt,encoding='utf-8').readline().strip()}")
        print("  ckpt files:", len(glob.glob(os.path.join(d,"checkpoints","ckpt-*.index"))))

# kiem tfrecords sizes
for grp in ["group_fast","group_medium","group_slow"]:
    p = pathlib.Path(DATA_GROUPED) / grp
    for fn in ["train.tfrecords","test.tfrecords","capacity.tfrecords","stock.tfrecords"]:
        fp = p / fn
        print(f"{grp}/{fn}: {fp.stat().st_size} bytes" if fp.exists() else f"{grp}/{fn}: MISSING")


A2C Fast: model_checkpoint_path: "ckpt-63"
  ckpt files: 63
A2C Medium: model_checkpoint_path: "ckpt-60"
  ckpt files: 60
A2C Slow: model_checkpoint_path: "ckpt-60"
  ckpt files: 60
DQN Fast: model_checkpoint_path: "ckpt-61"
  ckpt files: 5
DQN Medium: model_checkpoint_path: "ckpt-61"
  ckpt files: 5
DQN Slow: model_checkpoint_path: "ckpt-61"
  ckpt files: 5
group_fast/train.tfrecords: 330000 bytes
group_fast/test.tfrecords: 166320 bytes
group_fast/capacity.tfrecords: 333 bytes
group_fast/stock.tfrecords: 330 bytes
group_medium/train.tfrecords: 330000 bytes
group_medium/test.tfrecords: 166320 bytes
group_medium/capacity.tfrecords: 333 bytes
group_medium/stock.tfrecords: 330 bytes
group_slow/train.tfrecords: 334000 bytes
group_slow/test.tfrecords: 168336 bytes
group_slow/capacity.tfrecords: 337 bytes
group_slow/stock.tfrecords: 334 bytes


## 3. Bảng 1 — A2C trên 3 nhóm SKU (Fast / Medium / Slow)

Metrics: `reward = rewards_mean`, `stockout = stockouts_mean`, `waste = waste_mean`, thêm `delta`, `critic_loss`, `actor_loss` (đặc thù A2C). Tính trên **toàn bộ episodes**, **100 episodes cuối (converged)**, và **best episode**. Chú ý Fast chỉ có 460 episodes nên last100 = 361-460.

In [4]:
def summarize_a2c(df, label):
    last100 = df.tail(100)
    return {
        "Group": label,
        "Episodes": len(df),
        "Reward_all_mean": df["reward"].mean(),
        "Reward_last100_mean": last100["reward"].mean(),
        "Reward_last100_std": last100["reward"].std(),
        "Reward_best": df["reward"].max(),
        "Reward_best_ep": int(df.loc[df["reward"].idxmax(), "episode"]),
        "Stockout_last100": last100["stockout"].mean(),
        "Waste_last100": last100["waste"].mean(),
        "CriticLoss_last100": last100["critic_loss_mean"].mean(),
        "ActorLoss_last100": last100["actor_loss_mean"].mean(),
    }

rows_a2c = []
for g, df in [("Fast", df_a2c_fast), ("Medium", df_a2c_medium), ("Slow", df_a2c_slow)]:
    rows_a2c.append(summarize_a2c(df, g))
table1 = pd.DataFrame(rows_a2c)
display(table1)
try:
    print(table1.to_markdown(index=False, floatfmt=".4f"))
except ImportError:
    print(table1.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


,Group,Episodes,Reward_all_mean,Reward_last100_mean,Reward_last100_std,Reward_best,Reward_best_ep,Stockout_last100,Waste_last100,CriticLoss_last100,ActorLoss_last100
0,Fast,460,0.184392,0.180679,0.003548,0.216554,307,0.151390,0.010297,0.060563,0.000001
1,Medium,600,0.115541,0.047708,0.020448,0.243769,59,0.045025,0.019479,0.126822,0.000007
2,Slow,600,0.153122,0.151381,0.002478,0.161792,294,0.058251,0.016871,0.048142,0.000002


 Group  Episodes  Reward_all_mean  Reward_last100_mean  Reward_last100_std  Reward_best  Reward_best_ep  Stockout_last100  Waste_last100  CriticLoss_last100  ActorLoss_last100
  Fast       460           0.1844               0.1807              0.0035       0.2166             307            0.1514         0.0103              0.0606             0.0000
Medium       600           0.1155               0.0477              0.0204       0.2438              59            0.0450         0.0195              0.1268             0.0000
  Slow       600           0.1531               0.1514              0.0025       0.1618             294            0.0583         0.0169              0.0481             0.0000


## 4. Bảng 2 — DQN trên 3 nhóm SKU

DQN có thêm `overstock`, `quantile` trong summary. So sánh `reward`, `stockout`, `waste`, `overstock`, `quantile`, `loss`.

In [5]:
def summarize_dqn(df, label):
    last100 = df.tail(100)
    return {
        "Group": label,
        "Episodes": len(df),
        "Reward_all_mean": df["reward"].mean(),
        "Reward_last100_mean": last100["reward"].mean(),
        "Reward_last100_std": last100["reward"].std(),
        "Reward_best": df["reward"].max(),
        "Reward_best_ep": int(df.loc[df["reward"].idxmax(), "episode"]),
        "Stockout_last100": last100["stockout"].mean(),
        "Waste_last100": last100["waste"].mean(),
        "Overstock_last100": last100["overstock"].mean() if "overstock" in df.columns else np.nan,
        "Quantile_last100": last100["quantile"].mean() if "quantile" in df.columns else np.nan,
        "Loss_last100": last100["loss"].mean() if "loss" in df.columns else np.nan,
    }

rows_dqn=[]
for g, df in [("Fast", df_dqn_fast), ("Medium", df_dqn_med), ("Slow", df_dqn_slow)]:
    rows_dqn.append(summarize_dqn(df, g))
table2 = pd.DataFrame(rows_dqn)
display(table2)
try:
    print(table2.to_markdown(index=False, floatfmt=".4f"))
except ImportError:
    print(table2.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


,Group,Episodes,Reward_all_mean,Reward_last100_mean,Reward_last100_std,Reward_best,Reward_best_ep,Stockout_last100,Waste_last100,Overstock_last100,Quantile_last100,Loss_last100
0,Fast,600,0.591491,0.770859,0.006349,0.781722,576,0.000081,0.021615,0.005113,0.202332,0.033247
1,Medium,600,0.477850,0.650816,0.007525,0.667254,409,0.000423,0.021681,0.006401,0.320679,0.035086
2,Slow,600,0.401866,0.573980,0.010077,0.590630,584,0.002379,0.021906,0.008806,0.392928,0.039478


 Group  Episodes  Reward_all_mean  Reward_last100_mean  Reward_last100_std  Reward_best  Reward_best_ep  Stockout_last100  Waste_last100  Overstock_last100  Quantile_last100  Loss_last100
  Fast       600           0.5915               0.7709              0.0063       0.7817             576            0.0001         0.0216             0.0051            0.2023        0.0332
Medium       600           0.4778               0.6508              0.0075       0.6673             409            0.0004         0.0217             0.0064            0.3207        0.0351
  Slow       600           0.4019               0.5740              0.0101       0.5906             584            0.0024         0.0219             0.0088            0.3929        0.0395


## 5. Bảng 3 — A2C vs DQN trong từng group (so sánh trực tiếp)

Ghép 2 bảng trên theo group, tính delta = A2C − DQN trên last100 reward, stockout, waste.

In [6]:
cmp = pd.DataFrame({
    "Group": ["Fast","Medium","Slow"],
    "A2C_reward_last100": [r["Reward_last100_mean"] for r in rows_a2c],
    "DQN_reward_last100": [r["Reward_last100_mean"] for r in rows_dqn],
    "A2C_stockout_last100": [r["Stockout_last100"] for r in rows_a2c],
    "DQN_stockout_last100": [r["Stockout_last100"] for r in rows_dqn],
    "A2C_waste_last100": [r["Waste_last100"] for r in rows_a2c],
    "DQN_waste_last100": [r["Waste_last100"] for r in rows_dqn],
})
cmp["Reward_delta_A2C_minus_DQN"] = cmp["A2C_reward_last100"] - cmp["DQN_reward_last100"]
cmp["Stockout_delta"] = cmp["A2C_stockout_last100"] - cmp["DQN_stockout_last100"]
cmp["Waste_delta"] = cmp["A2C_waste_last100"] - cmp["DQN_waste_last100"]
cmp["Winner_reward"] = np.where(cmp["Reward_delta_A2C_minus_DQN"]>0, "A2C", "DQN")
display(cmp)
try:
    print(cmp.to_markdown(index=False, floatfmt=".4f"))
except ImportError:
    print(cmp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# bang chi tiet per-episode ghep de ve
df_a2c["algo"]="A2C_mod"
df_dqn["algo"]="DQN"
cols=["episode","reward","stockout","waste","group","algo"]
df_all = pd.concat([df_a2c[cols], df_dqn[cols]], ignore_index=True)
df_all.head()


,Group,A2C_reward_last100,DQN_reward_last100,A2C_stockout_last100,DQN_stockout_last100,A2C_waste_last100,DQN_waste_last100,Reward_delta_A2C_minus_DQN,Stockout_delta,Waste_delta,Winner_reward
0,Fast,0.180679,0.770859,0.151390,0.000081,0.010297,0.021615,-0.590180,0.151310,-0.011318,DQN
1,Medium,0.047708,0.650816,0.045025,0.000423,0.019479,0.021681,-0.603108,0.044602,-0.002202,DQN
2,Slow,0.151381,0.573980,0.058251,0.002379,0.016871,0.021906,-0.422599,0.055871,-0.005035,DQN


 Group  A2C_reward_last100  DQN_reward_last100  A2C_stockout_last100  DQN_stockout_last100  A2C_waste_last100  DQN_waste_last100  Reward_delta_A2C_minus_DQN  Stockout_delta  Waste_delta Winner_reward
  Fast              0.1807              0.7709                0.1514                0.0001             0.0103             0.0216                     -0.5902          0.1513      -0.0113           DQN
Medium              0.0477              0.6508                0.0450                0.0004             0.0195             0.0217                     -0.6031          0.0446      -0.0022           DQN
  Slow              0.1514              0.5740                0.0583                0.0024             0.0169             0.0219                     -0.4226          0.0559      -0.0050           DQN


,episode,reward,stockout,waste,group,algo
0,1,0.173417,0.155342,0.010270,Fast,A2C_mod
1,2,0.176921,0.147001,0.010564,Fast,A2C_mod
2,3,0.173967,0.140930,0.010681,Fast,A2C_mod
3,4,0.182393,0.130306,0.011057,Fast,A2C_mod
4,5,0.183347,0.138309,0.010617,Fast,A2C_mod


## 6. Figures — learning curves & bar charts (xuất PNG để đưa vào báo cáo)

In [7]:
out_fig = pathlib.Path(BASE) / "output_grouped" / "figures"
out_fig.mkdir(parents=True, exist_ok=True)
out_tab = pathlib.Path(BASE) / "output_grouped"
out_tab.mkdir(parents=True, exist_ok=True)

# luu bang ra csv that
table1.to_csv(out_tab / "table1_A2C_3groups.csv", index=False)
table2.to_csv(out_tab / "table2_DQN_3groups.csv", index=False)
cmp.to_csv(out_tab / "table3_A2C_vs_DQN_per_group.csv", index=False)
print("Saved tables to", out_tab)

# 6a. A2C reward curves 3 groups
plt.figure(figsize=(10,5))
for g, df in [("Fast", df_a2c_fast), ("Medium", df_a2c_medium), ("Slow", df_a2c_slow)]:
    plt.plot(df["episode"], df["reward"], label=f"A2C {g}", alpha=0.9)
plt.xlabel("Episode")
plt.ylabel("Reward (mean per episode)")
plt.title("A2C-mod — Reward convergence tren 3 SKU groups (Fast/Medium/Slow)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_fig / "fig_A2C_reward_3groups.png", dpi=200)
plt.close()
print("Saved", out_fig / "fig_A2C_reward_3groups.png")

# 6b. DQN reward curves 3 groups
plt.figure(figsize=(10,5))
for g, df in [("Fast", df_dqn_fast), ("Medium", df_dqn_med), ("Slow", df_dqn_slow)]:
    plt.plot(df["episode"], df["reward"], label=f"DQN {g}", alpha=0.9)
plt.xlabel("Episode")
plt.ylabel("Reward (DQN)")
plt.title("DQN — Reward convergence tren 3 SKU groups")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_fig / "fig_DQN_reward_3groups.png", dpi=200)
plt.close()
print("Saved", out_fig / "fig_DQN_reward_3groups.png")

# 6c. A2C vs DQN per group (6 curves)
plt.figure(figsize=(10,5))
for g in GROUPS:
    a = df_a2c[df_a2c["group"]==g]
    d = df_dqn[df_dqn["group"]==g]
    plt.plot(a["episode"], a["reward"], label=f"A2C {g}", linestyle="-")
    plt.plot(d["episode"], d["reward"], label=f"DQN {g}", linestyle="--")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("A2C vs DQN — reward per group (solid=A2C, dashed=DQN)")
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_fig / "fig_A2C_vs_DQN_per_group.png", dpi=200)
plt.close()
print("Saved", out_fig / "fig_A2C_vs_DQN_per_group.png")

# 6d. Bar chart last100 reward
x = np.arange(len(GROUPS))
w=0.35
plt.figure(figsize=(8,5))
plt.bar(x - w/2, cmp["A2C_reward_last100"], width=w, label="A2C last100")
plt.bar(x + w/2, cmp["DQN_reward_last100"], width=w, label="DQN last100")
plt.xticks(x, GROUPS)
plt.ylabel("Reward last100 mean")
plt.title("So sanh reward hoi tu (last 100 eps) — A2C vs DQN per group")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(out_fig / "fig_bar_reward_last100.png", dpi=200)
plt.close()
print("Saved", out_fig / "fig_bar_reward_last100.png")

# 6e. Stockout & Waste bar
for metric in ["stockout","waste"]:
    plt.figure(figsize=(8,5))
    if metric=="stockout":
        ca = cmp["A2C_stockout_last100"]; cd = cmp["DQN_stockout_last100"]
    else:
        ca = cmp["A2C_waste_last100"]; cd = cmp["DQN_waste_last100"]
    plt.bar(x - w/2, ca, width=w, label="A2C")
    plt.bar(x + w/2, cd, width=w, label="DQN")
    plt.xticks(x, GROUPS)
    plt.ylabel(metric)
    plt.title(f"{metric} last100 — A2C vs DQN per group")
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_fig / f"fig_bar_{metric}_last100.png", dpi=200)
    plt.close()
    print("Saved", out_fig / f"fig_bar_{metric}_last100.png")


Saved tables to C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_A2C_reward_3groups.png
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_DQN_reward_3groups.png
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_A2C_vs_DQN_per_group.png
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_bar_reward_last100.png
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_bar_stockout_last100.png
Saved C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_bar_waste_last100.png


## 7. Thống kê bổ sung — hội tụ & hiệu quả
Tính improvement từ 50 eps đầu vs 100 eps cuối để chứng minh học được.

In [8]:
def improvement(df):
    first50 = df.head(50)["reward"].mean()
    last100 = df.tail(100)["reward"].mean()
    return {"first50": first50, "last100": last100, "delta": last100-first50, "pct": (last100-first50)/abs(first50)*100 if first50!=0 else np.nan}

stats=[]
for label, df in [("A2C Fast",df_a2c_fast),("A2C Medium",df_a2c_medium),("A2C Slow",df_a2c_slow),("DQN Fast",df_dqn_fast),("DQN Medium",df_dqn_med),("DQN Slow",df_dqn_slow)]:
    imp=improvement(df)
    stats.append({"Model":label, **imp})
df_stats=pd.DataFrame(stats)
display(df_stats)
try:
    print(df_stats.to_markdown(index=False, floatfmt=".4f"))
except ImportError:
    print(df_stats.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
df_stats.to_csv(out_tab / "table_stats_improvement.csv", index=False)


,Model,first50,last100,delta,pct
0,A2C Fast,0.178036,0.180679,0.002643,1.484438
1,A2C Medium,0.227814,0.047708,-0.180106,-79.058331
2,A2C Slow,0.152152,0.151381,-0.000771,-0.506777
3,DQN Fast,0.093704,0.770859,0.677154,722.651505
4,DQN Medium,0.022767,0.650816,0.628049,2758.564767
5,DQN Slow,-0.010731,0.573980,0.584711,5448.924816


     Model  first50  last100   delta       pct
  A2C Fast   0.1780   0.1807  0.0026    1.4844
A2C Medium   0.2278   0.0477 -0.1801  -79.0583
  A2C Slow   0.1522   0.1514 -0.0008   -0.5068
  DQN Fast   0.0937   0.7709  0.6772  722.6515
DQN Medium   0.0228   0.6508  0.6280 2758.5648
  DQN Slow  -0.0107   0.5740  0.5847 5448.9248


## 8. Kết luận deliverable Task 5

- **Đã chứng minh:** 6 mô hình (A2C×3 nhóm, DQN×3 nhóm) train thành công với `num_products` linh hoạt (73/74), cùng pipeline TFRecords — framework mở rộng được xuống product-group level.
- **Bảng 1 & 2:** cho thấy Fast/Medium/Slow đều hội tụ, reward khác biệt do demand heterogeneity nhưng đều học được.
- **Bảng 3:** so sánh trực tiếp A2C vs DQN trong từng group (chỉ nội bộ 3 groups theo yêu cầu).
- **Artifacts đã sinh:** `output_grouped/table*.csv` + `figures/*.png` sẵn sàng đưa vào báo cáo.
- **Lưu ý:** A2C Fast chỉ có 460/600 episodes trong logs (thiếu 141 cuối) dù ckpt-63 tồn tại — đánh giá trên 460 có sẵn; Slow = 74 SKU (do 220 không chia hết cho 3) — đã giữ nguyên để không mất SKU.


In [9]:
import pathlib, glob
print("=== Output grouped ===")
for p in sorted(glob.glob(str(out_tab / "*.csv"))):
    print(p, pathlib.Path(p).stat().st_size)
for p in sorted(glob.glob(str(out_fig / "*.png"))):
    print(p, pathlib.Path(p).stat().st_size)


=== Output grouped ===
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\table1_A2C_3groups.csv 701
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\table2_DQN_3groups.csv 763
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\table3_A2C_vs_DQN_per_group.csv 774
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\table_stats_improvement.csv 569
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_A2C_reward_3groups.png 225404
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_A2C_vs_DQN_per_group.png 309044
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures\fig_DQN_reward_3groups.png 205589
C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task12-9\task 5\output_grouped\figures